In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Preprocessing

In [5]:
true_df = pd.read_csv('/content/True.csv')
fake_df = pd.read_csv('/content/Fake.csv')

In [6]:
print(true_df.head())
print(fake_df.head())

                                               title  ...                date
0  As U.S. budget fight looms, Republicans flip t...  ...  December 31, 2017 
1  U.S. military to accept transgender recruits o...  ...  December 29, 2017 
2  Senior U.S. Republican senator: 'Let Mr. Muell...  ...  December 31, 2017 
3  FBI Russia probe helped by Australian diplomat...  ...  December 30, 2017 
4  Trump wants Postal Service to charge 'much mor...  ...  December 29, 2017 

[5 rows x 4 columns]
                                               title  ...               date
0   Donald Trump Sends Out Embarrassing New Year’...  ...  December 31, 2017
1   Drunk Bragging Trump Staffer Started Russian ...  ...  December 31, 2017
2   Sheriff David Clarke Becomes An Internet Joke...  ...  December 30, 2017
3   Trump Is So Obsessed He Even Has Obama’s Name...  ...  December 29, 2017
4   Pope Francis Just Called Out Donald Trump Dur...  ...  December 25, 2017

[5 rows x 4 columns]


In [7]:
true_df['label'] = 0
fake_df['label'] = 1

1. True News = 0
2. Fake News = 1

In [8]:
print(true_df.head())
print(fake_df.head())

                                               title  ... label
0  As U.S. budget fight looms, Republicans flip t...  ...     0
1  U.S. military to accept transgender recruits o...  ...     0
2  Senior U.S. Republican senator: 'Let Mr. Muell...  ...     0
3  FBI Russia probe helped by Australian diplomat...  ...     0
4  Trump wants Postal Service to charge 'much mor...  ...     0

[5 rows x 5 columns]
                                               title  ... label
0   Donald Trump Sends Out Embarrassing New Year’...  ...     1
1   Drunk Bragging Trump Staffer Started Russian ...  ...     1
2   Sheriff David Clarke Becomes An Internet Joke...  ...     1
3   Trump Is So Obsessed He Even Has Obama’s Name...  ...     1
4   Pope Francis Just Called Out Donald Trump Dur...  ...     1

[5 rows x 5 columns]


In [15]:
true_df.isnull().sum()

,0
title,0
text,0
subject,0
date,0
label,0


In [16]:
fake_df.isnull().sum()

,0
title,0
text,0
subject,0
date,0
label,0


In [11]:
print(true_df.shape)
print(fake_df.shape)

(21417, 5)
(23481, 5)


In [12]:
print(true_df['date'])

0        December 31, 2017 
1        December 29, 2017 
2        December 31, 2017 
3        December 30, 2017 
4        December 29, 2017 
                ...        
21412      August 22, 2017 
21413      August 22, 2017 
21414      August 22, 2017 
21415      August 22, 2017 
21416      August 22, 2017 
Name: date, Length: 21417, dtype: object


In [13]:
print(true_df.columns)
print(fake_df.columns)

Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')
Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')


In [17]:
news_dataset = pd.concat([true_df, fake_df], axis=0)

In [18]:
news_dataset = news_dataset.sample(frac = 1, random_state=42).reset_index(drop=True)

In [19]:
news_dataset["id"] = news_dataset.index

In [20]:
news_dataset.head()

,title,text,subject,date,label,id
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1,0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1,1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1,2
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0,3
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1,4


In [21]:
cols = ["id"] + [col for col in news_dataset.columns if col != "id"]
news_dataset = news_dataset[cols]

In [22]:
news_dataset.head()

,id,title,text,subject,date,label
0,0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1
1,1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1
2,2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1
3,3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0
4,4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1


In [52]:
news_dataset.to_csv('F:\Machine Learning Projects\Project 4 - Fake News Prediction using Machine Learning\news_dataset.csv', index=False)

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_30045/4277763696.py:1: SyntaxWarning: invalid escape sequence '\M'
  news_dataset.to_csv('F:\Machine Learning Projects\Project 4 - Fake News Prediction using Machine Learning\news_dataset.csv', index=False)


merging the news title and news text

In [26]:
news_dataset['content'] = news_dataset['title'] + ' ' + news_dataset['text']

In [27]:
news_dataset.head()

,id,title,text,subject,date,label,content
0,0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1,BREAKING: GOP Chairman Grassley Has Had Enoug...
1,1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1,Failed GOP Candidates Remembered In Hilarious...
2,2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1,Mike Pence’s New DC Neighbors Are HILARIOUSLY...
3,3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0,California AG pledges to defend birth control ...
4,4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1,AZ RANCHERS Living On US-Mexico Border Destroy...


In [28]:
# Separating the data and label
X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

In [29]:
print(X)
print(Y)

          id  ...                                            content
0          0  ...   BREAKING: GOP Chairman Grassley Has Had Enoug...
1          1  ...   Failed GOP Candidates Remembered In Hilarious...
2          2  ...   Mike Pence’s New DC Neighbors Are HILARIOUSLY...
3          3  ...  California AG pledges to defend birth control ...
4          4  ...  AZ RANCHERS Living On US-Mexico Border Destroy...
...      ...  ...                                                ...
44893  44893  ...  Nigeria says U.S. agrees delayed $593 million ...
44894  44894  ...  Boiler Room #62 – Fatal Illusions Tune in to t...
44895  44895  ...  ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...
44896  44896  ...  Republican tax plan would deal financial hit t...
44897  44897  ...  U.N. refugee commissioner says Australia must ...

[44898 rows x 6 columns]
0        1
1        1
2        1
3        0
4        1
        ..
44893    0
44894    1
44895    1
44896    0
44897    0
Name: label, Length: 4489

Stemming:

Stemming is the process of reducing a word to its Root word

In [30]:
port_stem = PorterStemmer()

In [31]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content
#

In [32]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [33]:
print(news_dataset['content'])

0        break gop chairman grassley enough demand trum...
1        fail gop candid rememb hilari mock eulog video...
2        mike penc new dc neighbor hilari troll homopho...
3        california ag pledg defend birth control insur...
4        az rancher live us mexico border destroy nanci...
                               ...                        
44893    nigeria say u agre delay million fighter plane...
44894    boiler room fatal illus tune altern current ra...
44895    atheist sue governor texa display capitol grou...
44896    republican tax plan would deal financi hit u u...
44897    u n refuge commission say australia must stop ...
Name: content, Length: 44898, dtype: object


In [34]:
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [35]:
print(X)

['break gop chairman grassley enough demand trump jr testimoni donald trump white hous chao tri cover russia problem mount hour refus acknowledg problem surround fake news hoax howev fact bear thing differ seem crack congression public leadership chuck grassley r iowa head senat judiciari committe fed demand donald trump jr former trump campaign manag paul manafort testifi committe regard infam shadi meet donald trump shadi russian lawyer promis dirt democrat presidenti nomine hillari clinton fact inform due well demand send signal team trump notabl fire special counsel robert mueller circumst despit fact seem seem trump white hous lay groundwork speak speak tweet regard grassley warn also anyon think senat grassley rest senat seriou need look warn alreadi given trump jr manafort either follow order serv subpoena forc compli refus held contempt congress carri seriou jail time even cruel craven creatur within gop sick donald trump corrupt scandal ridden white hous angri stage hostil tak

In [37]:
print(Y)

[1 1 1 ... 1 0 0]


In [38]:
Y.shape

(44898,)

In [39]:
# Converting the textual data to numerical data

vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [40]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6904081 stored elements and shape (44898, 89868)>
  Coords	Values
  (0, 570)	0.06055486453372135
  (0, 2286)	0.04253552689623891
  (0, 2301)	0.025666202113816185
  (0, 3031)	0.063093661763644
  (0, 3388)	0.047259753130124114
  (0, 6574)	0.07041583382179394
  (0, 7853)	0.0898677958851911
  (0, 7886)	0.059913473329118644
  (0, 9454)	0.048097596645320734
  (0, 10324)	0.08231690448262517
  (0, 11174)	0.03296114815015033
  (0, 11631)	0.04919300578107092
  (0, 12522)	0.05138750652591612
  (0, 12642)	0.06680808165549404
  (0, 13496)	0.06811316396970238
  (0, 13713)	0.07112856852592032
  (0, 13991)	0.053000278097355684
  (0, 14098)	0.037972261733148126
  (0, 14836)	0.0869210456407381
  (0, 14928)	0.06842646468851935
  (0, 15149)	0.04165264736324387
  (0, 15153)	0.051876038877662234
  (0, 15333)	0.08494945735141231
  (0, 15663)	0.056535937666043194
  (0, 15765)	0.029199968119368196
  :	:
  (44897, 77773)	0.03757368913834899
  (44897,

Splitting the dataset to training & test data

In [41]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify= Y, random_state=2)

Applying Logistic Regression Model

In [43]:
model = LogisticRegression()

In [44]:
model.fit(X_train, Y_train)

LogisticRegression()

Evaluating accuracy score

In [45]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [46]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9917033242385433


In [47]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [48]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9876391982182628


Building a Predicition system

In [59]:
X_new = X_test[4]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('The news is Real')
else:
  print('The news is Fake')

[0]
The news is Real


In [60]:
print(Y_test[4])

0
